# Gradient Validation and Visualization

This notebook demonstrates the end-to-end differentiability of the `gyaradax` solver. We define a simple loss function based on the simulation state after a single time step, compute its gradient with respect to the initial distribution function using JAX's reverse-mode autodiff (`jax.grad`), and validate it against numerical finite differences.

In [ ]:
import sys
import time
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

sys.path.append("..")
from gyaradax import load_geometry, GKParams, default_state, gkstep_single
from gyaradax.plot_utils import plot_nd

# Enforce strict precision
jax.config.update("jax_enable_x64", True)

# Plotting style
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 14,
        "figure.titlesize": 16,
    }
)

## 1. Setup Simulation Environment

We load a reference geometry (e.g., `iteration_13_Lin`) and initialize a random distribution function `df0` to ensure gradients are non-trivial across all dimensions.

In [ ]:
BASE = "/restricteddata/ukaea/gyrokinetics/raw/iteration_13_Lin"
geom = load_geometry(BASE)

shape = (
    len(geom["intvp"]),
    len(geom["intmu"]),
    len(geom["ints"]),
    len(geom["kxrh"]),
    len(geom["krho"]),
)
print(f"Loaded geometry with phase space shape: {shape}")

# Create a random initial state
key = jax.random.PRNGKey(42)
df0_real = jax.random.normal(key, shape, dtype=jnp.float64)
df0_imag = jax.random.normal(jax.random.PRNGKey(43), shape, dtype=jnp.float64)
df0 = df0_real + 1j * df0_imag

params = GKParams(dt=0.01, naverage=40, non_linear=False)
state = default_state()

## 2. Define Differentiable Loss Function

We define a scalar loss $L = \sum |f_{t+1}|^2$ and use `jax.grad` to compute $\nabla_{f_0} L$. For complex variables in JAX, `jax.grad` returns the gradient such that `df0 -= lr * grad` performs steepest descent.

In [ ]:
def loss_fn(df_input):
    # 1. Step the solver forward one step
    next_df, _, _ = gkstep_single(df_input, geom, params, state)

    # 2. Compute a scalar real loss (e.g., energy / sum of squares)
    return jnp.sum(jnp.abs(next_df) ** 2)


# JIT compile the loss and its gradient function
loss_jitted = jax.jit(loss_fn)
grad_fn_jitted = jax.jit(jax.grad(loss_fn))

print("Compiling loss and gradient functions...")
t0 = time.time()
l_val = loss_jitted(df0)
print(f"Loss compiled and executed in {time.time() - t0:.2f}s | Value: {l_val:.4e}")

t0 = time.time()
analytical_grad = grad_fn_jitted(df0)
analytical_grad.block_until_ready()
print(f"Gradient compiled and executed in {time.time() - t0:.2f}s")
print(f"Analytical Gradient Shape: {analytical_grad.shape}")

## 3. Visualize the Gradient Array

We can inspect the spatial and velocity structure of the gradient $\nabla_{f_0} L$. The gradient dictates how modifying the initial state at a specific phase-space coordinate affects the downstream loss.

In [ ]:
# Plot slices of the gradient magnitude using our n-dimensional plotting utility
fig = plot_nd(analytical_grad, title="Analytical Gradient Magnitude $|\nabla_{f_0} L|$")
plt.show()

## 4. Finite Difference Validation

To definitively prove that the reverse-mode AD gradient is correct, we compute the gradient via central finite differences for a random subset of coordinates and compare them. 

For a complex function $L(z)$, the JAX `grad` convention gives $\nabla_z L = \frac{\partial L}{\partial x} - i \frac{\partial L}{\partial y}$. We will validate the real part independently.

In [ ]:
epsilon = 1e-6
num_samples = 50

# Select random indices across the 5D grid
np.random.seed(123)
flat_indices = np.random.choice(df0.size, num_samples, replace=False)
indices = np.unravel_index(flat_indices, df0.shape)

fd_grads_real = []
ad_grads_real = []

print(f"Computing finite differences for {num_samples} samples...")
t0 = time.time()
for i in range(num_samples):
    idx = tuple(ind[i] for ind in indices)

    # Base values
    val = df0[idx]

    # Perturb Real Part
    df_plus = df0.at[idx].set(val + epsilon)
    df_minus = df0.at[idx].set(val - epsilon)

    loss_plus = loss_jitted(df_plus)
    loss_minus = loss_jitted(df_minus)

    fd_grad_r = (loss_plus - loss_minus) / (2 * epsilon)

    # JAX grad convention for complex: df/dz = df/dx - i df/dy
    # Therefore, the real part of the AD gradient corresponds to the perturbation of the real part.
    ad_grad_r = jnp.real(analytical_grad[idx])

    fd_grads_real.append(float(fd_grad_r))
    ad_grads_real.append(float(ad_grad_r))

print(f"Done in {time.time() - t0:.2f}s")

fd_grads_real = np.array(fd_grads_real)
ad_grads_real = np.array(ad_grads_real)

# Calculate relative error
rel_errors = np.abs(ad_grads_real - fd_grads_real) / (np.abs(ad_grads_real) + 1e-30)
max_err = np.max(rel_errors)
mean_err = np.mean(rel_errors)

print("\nValidation Results:")
print(f"Max Relative Error:  {max_err:.4e}")
print(f"Mean Relative Error: {mean_err:.4e}")

## 5. Parity Scatter Plot
A tight alignment along the identity line $y=x$ indicates perfect agreement between the automatic differentiation and the finite difference approximation.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(fd_grads_real, ad_grads_real, alpha=0.7, edgecolors="k", s=50)

# Identity line
min_val = min(np.min(fd_grads_real), np.min(ad_grads_real))
max_val = max(np.max(fd_grads_real), np.max(ad_grads_real))
plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="Identity (y=x)")

plt.title("Analytical vs. Numerical Gradients (Real Part)")
plt.xlabel("Finite Difference Gradient")
plt.ylabel("JAX Autodiff Gradient")
plt.grid(True, alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()